In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
# Point these at your own data / weights, or set the env vars before launching
# Jupyter.
import os

# Ultralytics dataset root + its data yaml (det_seg_dataset)
DATASET_ROOT = os.environ.get("FLATFISH_DATASET_ROOT", "./data/det_seg_dataset")
DATA_YAML = os.environ.get("FLATFISH_YOLO_DATA_YAML", f"{DATASET_ROOT}/fishdataset.yaml")
# Trained YOLO weights
WEIGHTS = os.environ.get("FLATFISH_YOLO_WEIGHTS", "./fishdetect_result/train/weights/best.pt")
# Misc dataset root used by the label-shuffling / csv cells
DATA_ROOT = os.environ.get("FLATFISH_DATA_ROOT", "./data")
# Single image for a smoke-test prediction
SAMPLE_IMAGE = os.environ.get("FLATFISH_SAMPLE_IMAGE", f"{DATA_ROOT}/sample.jpg")
# mmdetection .pkl dumps, for the baseline comparison cells
MMDET_RESULT_DIR = os.environ.get("FLATFISH_MMDET_RESULT_DIR", "./mmdet_result")
# ───────────────────────────────────────────────────────────────────────────────


In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n.pt") 

In [ ]:
model.train(data=DATA_YAML, epochs=20, batch=2, project = "fishdetect_result/", device=[0])

In [ ]:
from ultralytics import YOLO
import torch
from torchvision.ops import box_iou  # torchvision에서 IoU 계산

# 1. 모델 로드
model = YOLO(WEIGHTS)  # 또는 커스텀 모델

metrics = model.val(data=DATA_YAML, device=[0], workers=0)

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS)  # 또는 커스텀 모델

In [ ]:
model.names

In [ ]:
model.names[0] = 'fish'

new_names_dict = {
    0: 'fish'
}

model.names[0] = new_names_dict

In [ ]:
model.model.names = {0: 'fish'}


model.save('new_checkpoint.pt')


In [ ]:
model2 = YOLO('new_checkpoint.pt')


In [ ]:
#model(source=f"{DATASET_ROOT}/images/IMG_6499.png", save=True)

model2(source=SAMPLE_IMAGE, save=True)

In [ ]:
import os
import torch
from ultralytics import YOLO
from torchvision.ops import box_iou
from pathlib import Path
import cv2

# 1. 모델 로드
model = YOLO(WEIGHTS)

# 2. data.yaml 기준 디렉토리 설정
dataset_root = Path(DATASET_ROOT)  # data.yaml의 path 기준
val_images_dir = dataset_root / "images/val"
val_labels_dir = dataset_root / "labels/val"

total_iou = 0.0
count = 0

for img_path in val_images_dir.glob("*.png"):
    stem = img_path.stem
    label_path = val_labels_dir / f"{stem}.txt"
    if not label_path.exists():
        continue

    # 3. 이미지 크기
    img_cv = cv2.imread(str(img_path))
    h, w = img_cv.shape[:2]

    # 4. GT box 로드 (YOLO format → xyxy)
    gt_boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            cls, xc, yc, bw, bh = map(float, line.strip().split())
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            x2 = (xc + bw / 2) * w
            y2 = (yc + bh / 2) * h
            gt_boxes.append([x1, y1, x2, y2])

    if len(gt_boxes) == 0:
        continue

    gt_tensor = torch.tensor(gt_boxes, dtype=torch.float32)

    # 5. YOLOv8 예측
    result = model.predict(source=str(img_path), verbose=False)[0]
    if result.boxes.xyxy.numel() == 0:
        continue

    pred_boxes = result.boxes.xyxy
    scores = result.boxes.conf
    top_idx = scores.argmax()
    pred_box = pred_boxes[top_idx].unsqueeze(0)  # shape: (1, 4)

    # 6. IoU 계산
    ious = box_iou(pred_box.cpu(), gt_tensor)  # (1, N_gt)
    best_iou = ious.max().item()

    total_iou += best_iou
    count += 1

# 7. mIoU 출력
miou = total_iou / count if count > 0 else 0
print(f"\n🔹 mean IoU (YOLOv8 Top-1 prediction): {miou:.4f}")

In [ ]:
import pickle
import torch
from torchvision.ops import box_iou
from pycocotools.coco import COCO
import numpy as np

# 1. 예측 결과 로드
with open(f"{MMDET_RESULT_DIR}/faster_rcnn_output.pkl", 'rb') as f:
    outputs = pickle.load(f)  # list of dicts per image

In [ ]:
# 2. GT COCO annotation 로드
coco = COCO(f"{DATASET_ROOT}/annotations/instances_val2017.json")  # path 수정 필요


# 3. mIoU 계산
total_iou = 0.0
total_preds = 0

# 3. mIoU 계산 (top-1 prediction만 사용)
total_iou = 0.0
total_images = 0

for sample in outputs:
    img_id = sample['img_id']
    pred_scores = sample['pred_instances']['scores']
    pred_boxes = sample['pred_instances']['bboxes']

    if pred_boxes.numel() == 0:
        continue

    # Top-1 prediction 선택
    top_idx = pred_scores.argmax()
    pred_box = pred_boxes[top_idx].unsqueeze(0).float().cpu()  # (1, 4)

    # GT 박스 로드
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    gt_boxes = [ann['bbox'] for ann in anns if ann['iscrowd'] == 0]

    if len(gt_boxes) == 0:
        continue

    gt_boxes = [[x, y, x + w, y + h] for x, y, w, h in gt_boxes]
    gt_tensor = torch.tensor(gt_boxes, dtype=torch.float32)

    # IoU 계산
    ious = box_iou(pred_box, gt_tensor)  # shape: (1, N_gt)
    best_iou = ious.max().item()  # top-1 예측과 best GT IoU

    total_iou += best_iou
    total_images += 1

# 4. 최종 결과
miou = total_iou / total_images if total_images > 0 else 0
print(f"\n🔹 mean IoU (mIoU, Top-1 prediction only): {miou:.4f}")

In [ ]:
import pickle
import torch
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
import numpy as np

# 1. 예측 결과 로드
with open(f"{MMDET_RESULT_DIR}/solov2_output.pkl", 'rb') as f:
    outputs = pickle.load(f)  # list of dicts per image

# 2. GT COCO annotation 로드
coco = COCO(f"{DATASET_ROOT}/annotations/instances_val2017.json")

# 3. mIoU 계산
total_iou = 0.0
total_images = 0

for sample in outputs:
    img_id = sample['img_id']
    pred_scores = sample['pred_instances']['scores']
    pred_masks = sample['pred_instances']['masks']  # List of RLE dicts or numpy array

    if len(pred_masks) == 0:
        continue

    # Top-1 prediction 선택
    top_idx = pred_scores.argmax()
    pred_mask_rle = pred_masks[top_idx]

    # RLE 디코딩 (HxW binary mask)
    pred_mask = maskUtils.decode(pred_mask_rle).astype(np.uint8)

    # GT 마스크 로드
    ann_ids = coco.getAnnIds(imgIds=[img_id], iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    if len(anns) == 0:
        continue

    # GT 마스크들: COCO segmentation을 RLE로 변환 후 디코딩
    ious = []
    for ann in anns:
        gt_rle = maskUtils.frPyObjects(ann['segmentation'],
                                       coco.imgs[img_id]['height'],
                                       coco.imgs[img_id]['width'])
        gt_mask = maskUtils.decode(gt_rle).astype(np.uint8)

        # GT는 list of RLE일 수도 있음 → sum으로 병합
        if len(gt_mask.shape) == 3:
            gt_mask = np.any(gt_mask, axis=2).astype(np.uint8)

        # IoU 계산
        intersection = np.logical_and(pred_mask, gt_mask).sum()
        union = np.logical_or(pred_mask, gt_mask).sum()
        iou = intersection / union if union > 0 else 0.0
        ious.append(iou)

    if len(ious) > 0:
        total_iou += max(ious)  # best IoU 기준
        total_images += 1

# 4. 최종 결과
miou = total_iou / total_images if total_images > 0 else 0
print(f"\n🔹 mean IoU (mIoU, Top-1 prediction only): {miou:.4f}")


In [ ]:
import numpy as np
from scipy.spatial.distance import cdist

def pose_eval_simple(gt_poses, pred_poses):
    """
    This function evaluates the pose detection capability using L1 distance.
    Args:
        gt_poses: list or np.array, shape (N, 7) - ground truth poses
        pred_poses: list or np.array, shape (M, 7) - predicted poses
    Returns:
        mean_min_distance, mean_rate
    """
    gt_poses = np.array(gt_poses)
    pred_poses = np.array(pred_poses)

    all_min_dist = []
    all_rate = []

    # 1. cdist로 gt와 pred 간 거리 매트릭스 계산 (L1 거리 사용)
    distances = cdist(gt_poses, pred_poses, metric='cityblock')  # ← L1 거리: cityblock

    # 2. 매칭률 계산 (0.2 이하)
    rate = np.sum(np.any(distances <= 0.2, axis=1)) / len(gt_poses)
    all_rate.append(rate)

    # 3. 직접 L1 distance 계산 (g - g_pred의 절댓값 합)
    g = gt_poses[:, np.newaxis, :]    # (N, 1, 7)
    g_pred = pred_poses               # (M, 7)
    l1_distances = np.sum(np.abs(g - g_pred), axis=2)  # (N, M)

    min_distances = np.min(l1_distances, axis=1)  # (N, )
    mean_rate = np.mean(all_rate)

    return min_distances, mean_rate




In [ ]:
gt = [[0.5,0.5,0.5,0.1,0.1,0.1]]
pred = [[0.5,0.5,0.5,0.1,0.1,0.1]]

a, b = pose_eval_simple(gt, pred)
print(a)

In [ ]:
gt = [[-1,-1,-1,-3.14,3.14,3.14]]
pred = [[1,1,1,3.14,-3.14,-3.14]]

a, b = pose_eval_simple(gt, pred)
print(a)

In [ ]:
import os
import torch
from ultralytics import YOLO
from torchvision.ops import box_iou
from pathlib import Path
import cv2

# 1. 모델 로드
model = YOLO(WEIGHTS)

# 2. data.yaml 기준 디렉토리 설정
dataset_root = Path(f"{DATA_ROOT}/202507")  # data.yaml의 path 기준
val_images_dir = dataset_root / "total"
# val_labels_dir = dataset_root / "labels/val"


# 4. bbox 저장 디렉토리
bbox_vis_dir = dataset_root / "vis_bbox"
bbox_vis_dir.mkdir(parents=True, exist_ok=True)

total_iou = 0.0
count = 0

for img_path in val_images_dir.glob("*.JPG"):
    stem = img_path.stem
    

    img_cv = cv2.imread(str(img_path))
    h, w = img_cv.shape[:2]


    # 5. YOLOv8 예측
    result = model.predict(source=str(img_path), verbose=False)[0]

    xywhn = result.boxes.xywhn.detach().cpu().tolist()   # normalized center boxes
    xyxy = result.boxes.xyxy.detach().cpu().tolist()     # absolute xyxy

    for (x, y, w_n, h_n), (x1, y1, x2, y2) in zip(xywhn, xyxy):

        # 🔥 bbox 그리기 (Green box)
        cv2.rectangle(img_cv,
                        (int(x1), int(y1)), (int(x2), int(y2)),
                        (0, 255, 0), 3)  # (B,G,R)=green


    # 10. bbox 결과 이미지 저장
    bbox_img_path = bbox_vis_dir / f"{stem}_det.jpg"
    cv2.imwrite(str(bbox_img_path), img_cv)

    # # 6. 라벨 저장 디렉토리 생성
    # labels_dir = dataset_root / "labels"
    # labels_dir.mkdir(parents=True, exist_ok=True)
    # label_path = labels_dir / f"{stem}.txt"

    # # 7. 예측 박스 → YOLO 포맷(.txt)로 저장 (class=0)
    # if result.boxes is not None and len(result.boxes) > 0:
    #     xywhn = result.boxes.xywhn.detach().cpu().tolist()  # 정규화된 (x,y,w,h)
    #     with open(label_path, "w", encoding="utf-8") as f:
    #         for x, y, w_norm, h_norm in xywhn:
    #             f.write(f"0 {x:.6f} {y:.6f} {w_norm:.6f} {h_norm:.6f}\n")


In [ ]:
p = f"{DATA_ROOT}/detection/labels/train/"

filenames = os.listdir(p)

for filename in filenames:
    labels = []

    with open(os.path.join(p, filename), "r", encoding="utf-8") as f:
        for line in f:
            cls, xc, yc, w, h = map(float, line.strip().split())
            labels.append([int(cls), xc, yc, w, h])
    print(filename)
    #print(len(labels))

In [ ]:
import pandas as pd
import numpy as np
import shutil
import os

df = pd.read_csv(f"{DATA_ROOT}/2024/test.csv")

df

In [ ]:
filenames = np.array(df['name'])
filenames

In [ ]:
root_path = f"{DATA_ROOT}/detection/labels/val"
# filenames = os.listdir(root_path)

# filenames = [x.split('.')[0] for x in filenames]

for filename in filenames:
    shutil.move(os.path.join(root_path.replace('labels', 'images').replace('val','train'), filename), os.path.join(root_path.replace('labels', 'images'), filename))
    shutil.move(os.path.join(root_path.replace('labels', 'images').replace('val','train'), filename), os.path.join(root_path.replace('labels', 'images'), filename))